In [1]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

Custom datasets in PyTorch inherit from torch.utils.data.Dataset and must implement __len__ and __getitem__

For unlabeled images from a folder, build a custom class that lists files with os.listdir and loads them with PIL

For labeled images, torchvision.datasets.ImageFolder handles the directory structure automatically

The same dataset pattern works for audio, CSV data, or any file type by swapping the loading logic inside __getitem__

dataset[i]⟶(Xi​,yi​) thats what pytorch expects

X.shape == (batch_size, 784)
Y.shape == (batch_size,)
X.dtype == torch.float32
Y.dtype == torch.int64

In [2]:
class MyDataset(Dataset):
    
    def __init__(self, data_path:str):
        super().__init__()
        self.data = pd.read_csv(data_path).to_numpy()
        self.X, self.Y = self.data[:, 1:].astype(np.float32) / 255, self.data[:, 0].astype(np.int64)
        
    def __len__(self):
        return len(self.Y)
    
    def __getitem__(self, index):
        return (self.X[index],self.Y[index])

In [3]:
train_data = MyDataset("C:/python/torch_projects/data/mnist/fashion-mnist_train.csv")
test_data = MyDataset("C:/python/torch_projects/data/mnist/fashion-mnist_test.csv")

train_load, test_load = DataLoader(train_data, batch_size=64, shuffle=True), DataLoader(test_data, batch_size=64, shuffle=True)

print(len(train_data[0][0]))

784


batch_size=64 means the DataLoader groups 64 samples from your Dataset into one batch.

In [4]:
class MLP(nn.Module):
    
    def __init__(self,):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )
    
    def forward(self, X):
        
        logits = self.layers(X)
        return logits

In [ ]:
model = MLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [19]:
def train_loop(batch_size, dataset, loss_fn, optimizer, model):
    size = len(dataset.dataset)
    for batch, (X, y) in enumerate(dataset):
        
        pred = model(X)
        loss = loss_fn(pred, y)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataset, model, loss_fn):
    
    size = len(dataset.dataset)
    correct, test_loss = 0, 0
    with torch.no_grad():
        
        for (X, y) in dataset:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
                
    acc = correct / size
    
    print(f"Accuracy: {acc} | Loss: {test_loss}")

In [21]:
epochs = 20

for e in range(epochs):
    print(f"{e} Epoch:")
    train_loop(64, train_load, loss_fn, optimizer, model)
    test_loop(test_load, model, loss_fn)



0 Epoch:
loss: 0.944925  [   64/60000]
loss: 0.896638  [ 6464/60000]
loss: 0.777550  [12864/60000]
loss: 0.659152  [19264/60000]
loss: 0.917702  [25664/60000]
loss: 0.834822  [32064/60000]
loss: 0.970309  [38464/60000]
loss: 0.915994  [44864/60000]
loss: 0.782422  [51264/60000]
loss: 0.830883  [57664/60000]
Accuracy: 0.6796 | Loss: 130.29746252298355
1 Epoch:
loss: 0.905414  [   64/60000]
loss: 0.857085  [ 6464/60000]
loss: 0.736244  [12864/60000]
loss: 0.744552  [19264/60000]
loss: 0.721667  [25664/60000]
loss: 0.828555  [32064/60000]
loss: 0.885282  [38464/60000]
loss: 0.969829  [44864/60000]
loss: 0.715007  [51264/60000]
loss: 0.821100  [57664/60000]
Accuracy: 0.6879 | Loss: 126.00317853689194
2 Epoch:
loss: 0.643443  [   64/60000]
loss: 0.792415  [ 6464/60000]
loss: 0.750446  [12864/60000]
loss: 0.907132  [19264/60000]
loss: 0.819375  [25664/60000]
loss: 0.803627  [32064/60000]
loss: 0.678374  [38464/60000]
loss: 0.897442  [44864/60000]
loss: 0.853699  [51264/60000]
loss: 0.796914 